# Tiled Noise-Attack Detection on a Kaggle Dataset (2xT4)

**Goal of this notebook (the workflow you asked for):**

1. **Grab a dataset** that lives on Kaggle (animals images by default).
2. **Pull out 10 random images** from it.
3. **Run a noise (adversarial) attack** on each image so you can *see by eye* where it was attacked.
4. **Tile** each image into a **4x4 grid** with a reusable `tile_image(...)` function.
5. **Inspect every tile** and decide *was this tile attacked by noise?*  -> **flag** the attacked tiles.
6. **Compare** three things side by side so you can judge the detector:
   - the **original** image,
   - the **true** attacked region (ground truth, before detection),
   - the **detected** region (what the system flagged, after detection).

The point: catch noisy/attacked tiles **before training** so the model is not tricked by the
perturbation. Everything runs on Kaggle with **GPU T4 x2** and **Internet On**.

> Set the accelerator to **GPU T4 x2** and turn **Internet On** in the Kaggle notebook settings.

## 0. Install
`grad-cam` is the PyPI package for `jacobgil/pytorch-grad-cam` (imported as `pytorch_grad_cam`).

In [ ]:
!pip install -q grad-cam

## 1. Setup & GPU inventory
We list the GPUs so you can confirm both T4s are visible before running the heavy cells.

In [ ]:
# ============================================================
# CELL 1 - Setup & GPU inventory
# ============================================================
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision import models, transforms
from PIL import Image
import urllib.request, json, os, time
from concurrent.futures import ThreadPoolExecutor

N_GPU = torch.cuda.device_count()
DEVICES = [f"cuda:{i}" for i in range(N_GPU)] if N_GPU else ["cpu"]
print("GPUs found:", N_GPU)
for i in range(N_GPU):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")
print("Using devices:", DEVICES)

## 2. One ResNet50 per GPU
We keep an independent ResNet50 copy on each device so each GPU can work on its own slice of a
tile-batch in parallel. We also load human-readable ImageNet class names.

In [ ]:
# ============================================================
# CELL 2 - ResNet50 (ImageNet) on every GPU + labels
# ============================================================
def make_model(dev):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    return m.eval().to(dev)

MODELS = {d: make_model(d) for d in DEVICES}          # device -> model copy
print(f"Loaded {len(MODELS)} ResNet50 copies (one per device).")

url = "https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json"
idx2label = {int(k): v[1] for k, v in json.load(urllib.request.urlopen(url)).items()}

## 3. Preprocessing + the `tile_image` function
Images live in `[0,1]` pixel space; ImageNet normalization happens *inside* the model call so the
adversarial noise stays measured in **real pixels**.

`tile_image` is the reusable tiler you asked for: it cuts a `[1,3,SIZE,SIZE]` image into a
`GRID x GRID` stack of small tiles (row-major order). `untile` is its inverse for visualization.

In [ ]:
# ============================================================
# CELL 3 - Preprocessing, the tile function, helpers
# ============================================================
SIZE = 448            # working resolution per image
GRID = 4              # GRID x GRID tiles -> 4x4 = 16 tiles per image
TILE = SIZE // GRID   # tile side in pixels (112)

to_tensor = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])

_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def normalize(t):                       # normalize on the tensor's own device
    return (t - _mean.to(t.device)) / _std.to(t.device)

def load_image(path_or_url):
    # Load local path OR URL -> [1,3,SIZE,SIZE] float tensor in [0,1] (on CPU).
    if str(path_or_url).startswith("http"):
        fn = "/tmp/" + os.path.basename(path_or_url)
        if not os.path.exists(fn):
            urllib.request.urlretrieve(path_or_url, fn)
        path_or_url = fn
    img = Image.open(path_or_url).convert("RGB")
    return to_tensor(img).unsqueeze(0)

# ---- THE TILE FUNCTION ----
def tile_image(x, grid=GRID, tile=TILE):
    # [1,3,SIZE,SIZE] -> [grid*grid, 3, tile, tile]  (row-major tile order).
    p = x.unfold(2, tile, tile).unfold(3, tile, tile)        # [1,3,grid,grid,tile,tile]
    p = p.permute(0, 2, 3, 1, 4, 5).reshape(-1, 3, tile, tile)
    return p.contiguous()

def untile(tiles, grid=GRID, tile=TILE):
    # [grid*grid, C, tile, tile] -> [C, SIZE, SIZE] mosaic (inverse of tile_image).
    C = tiles.shape[1]
    g = tiles.reshape(grid, grid, C, tile, tile).permute(2, 0, 3, 1, 4)
    return g.reshape(C, grid * tile, grid * tile)

def tile_grid_to_full(per_tile_scalar, grid=GRID, tile=TILE):
    # [grid*grid] per-tile values -> [SIZE,SIZE] blocky heatmap for overlay.
    g = np.asarray(per_tile_scalar).reshape(grid, grid)
    return np.kron(g, np.ones((tile, tile)))

def to_np(t):  return t.squeeze().detach().cpu().permute(1, 2, 0).numpy()
def norm01(a):
    a = np.asarray(a, np.float32)
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

# quick sanity check that tiling is loss-less
_t = torch.rand(1, 3, SIZE, SIZE)
assert torch.allclose(untile(tile_image(_t)), _t.squeeze(0), atol=1e-6)
print(f"tile_image OK: {GRID}x{GRID} grid, each tile {TILE}x{TILE} px, {GRID*GRID} tiles/image")

## 4. Get the dataset & pull 10 random images
Point `FOLDER` at any Kaggle image dataset. By default we look for the **Animals-10** dataset
(`alessiocorrado99/animals10`). Add it via **+ Add Input** in the Kaggle sidebar.
If no folder is found, we fall back to downloading a handful of animal samples from the web.

In [ ]:
# ============================================================
# CELL 4 - Gather 10 random images from a Kaggle dataset (or web fallback)
# ============================================================
N_IMAGES = 10

CANDIDATE_FOLDERS = [
    "/kaggle/input/animals10/raw-img",
    "/kaggle/input/imagenette/imagenette2/val",
    "/kaggle/input/imagenette2/val",
]
FOLDER = next((f for f in CANDIDATE_FOLDERS if os.path.isdir(f)), "")

def gather_images(folder, n=N_IMAGES, seed=1):
    exts = (".jpg", ".jpeg", ".png", ".JPEG")
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.endswith(exts): paths.append(os.path.join(root, f))
    rng = np.random.default_rng(seed)
    return list(rng.choice(paths, size=min(n, len(paths)), replace=False)) if paths else []

if FOLDER:
    images = gather_images(FOLDER, n=N_IMAGES)
    print(f"Picked {len(images)} random images from {FOLDER}")
else:
    base = "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/"
    images = [base + n for n in [
        "n02099601_golden_retriever.JPEG", "n02123045_tabby.JPEG", "n02391049_zebra.JPEG",
        "n02129165_lion.JPEG", "n02129604_tiger.JPEG", "n02510455_giant_panda.JPEG",
        "n01518878_ostrich.JPEG", "n01806143_peacock.JPEG", "n01882714_koala.JPEG",
        "n02007558_flamingo.JPEG"]]
    print(f"No Kaggle folder found -- using {len(images)} downloaded animal samples")

# ---- Preview: actually SHOW the 10 images we will work on ----
ncol = 5
nrow = int(np.ceil(len(images) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3 * ncol, 3 * nrow))
for ax, p in zip(np.ravel(axes), images):
    ax.imshow(to_np(load_image(p)))             # load -> [0,1] tensor -> HxWx3
    ax.set_title(os.path.basename(str(p))[:22], fontsize=8)
    ax.axis("off")
for ax in np.ravel(axes)[len(images):]:         # hide any empty cells
    ax.axis("off")
fig.suptitle(f"The {len(images)} images we will attack & test", y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

## 5. The noise attack (FGSM), confined to random tiles
We run **FGSM** (one signed-gradient step) but zero the perturbation outside a randomly chosen set
of tiles. That gives us the **ground truth**: exactly which tiles carry the attack. `n_attacked`
controls how many of the 16 tiles get poisoned, `epsilon` controls the strength.

In [ ]:
# ============================================================
# CELL 5 - Per-tile FGSM attack (ground-truth generator)
# ============================================================
def tile_mask_from_ids(ids):
    # boolean [GRID*GRID] of attacked tiles -> pixel mask [1,1,SIZE,SIZE].
    m = torch.zeros(1, 1, SIZE, SIZE)
    for k in np.where(ids)[0]:
        r, c = divmod(int(k), GRID)
        m[..., r*TILE:(r+1)*TILE, c*TILE:(c+1)*TILE] = 1.0
    return m

def random_tile_mask(n_tiles=3, seed=None):
    rng = np.random.default_rng(seed)
    ids = np.zeros(GRID*GRID, bool)
    ids[rng.choice(GRID*GRID, size=n_tiles, replace=False)] = True
    return ids

def fgsm_attack(x, true_class, epsilon=0.08, tile_ids=None):
    # x:[1,3,SIZE,SIZE] in [0,1]. Confine noise to tile_ids if given. Returns adv image (CPU).
    d = DEVICES[0]
    xi = x.clone().detach().to(d).requires_grad_(True)
    out = MODELS[d](normalize(xi))
    loss = F.cross_entropy(out, torch.tensor([true_class], device=d))
    MODELS[d].zero_grad(); loss.backward()
    perturb = epsilon * xi.grad.sign()
    if tile_ids is not None:
        perturb = perturb * tile_mask_from_ids(tile_ids).to(d)
    return torch.clamp(xi + perturb, 0, 1).detach().cpu()

## 6. Batched prediction across both GPUs
Used to (a) pick the FGSM target class from the clean image and (b) check whether the global label
flipped after the attack. The tile batch is split across the available GPUs and run in threads
(CUDA ops release the GIL -> real parallelism).

In [ ]:
# ============================================================
# CELL 6 - Multi-GPU batched prediction
# ============================================================
def batched_predict(tiles_cpu, use_gpus=None):
    devs = use_gpus or DEVICES
    chunks = torch.chunk(tiles_cpu, len(devs), dim=0) if len(devs) > 1 else [tiles_cpu]
    devs = devs[:len(chunks)]
    def _run(args):
        d, ch = args
        with torch.no_grad():            # no_grad is thread-local -> must re-enter it in each worker
            p = MODELS[d](normalize(ch).to(d)).softmax(1)
            conf, idx = p.max(1)
        return idx.cpu().numpy(), conf.cpu().numpy()
    with ThreadPoolExecutor(max_workers=len(devs)) as ex:
        outs = list(ex.map(_run, zip(devs, chunks)))
    idx = np.concatenate([o[0] for o in outs]); conf = np.concatenate([o[1] for o in outs])
    return idx, conf

## 7. The blind per-tile detector
FGSM (and most pixel attacks) inject **high-frequency** noise. We score each tile by its
**high-frequency energy** = `mean(|tile - blur(tile)|)`. Attacked tiles spike above the image's
own tile-population baseline, so we flag any tile whose score exceeds **median + k*MAD** across that
image's 16 tiles. **No clean reference is needed** -- this is what makes it usable on images you did
not attack yourself (i.e. a real dataset you want to clean before training).

In [ ]:
# ============================================================
# CELL 7 - Blind high-frequency per-tile detector + metrics
# ============================================================
def _gaussian_kernel(sigma=1.0, ksize=5):
    ax = torch.arange(ksize) - ksize // 2
    g = torch.exp(-(ax**2) / (2*sigma**2)); g = g / g.sum()
    k = torch.outer(g, g)
    return k.view(1, 1, ksize, ksize).repeat(3, 1, 1, 1)   # depthwise, 3 ch

_GK = _gaussian_kernel()

def hf_energy_per_tile(tiles):
    # tiles:[B,3,h,w] in [0,1] -> [B] high-frequency energy score.
    k = _GK.to(tiles.device)
    blur = F.conv2d(tiles, k, padding=k.shape[-1]//2, groups=3)
    hf = (tiles - blur).abs().mean(dim=(1, 2, 3))
    return hf.cpu().numpy()

def detect_tiles(hf_scores, k=3.0):
    # Robust outlier flag: median + k*MAD. Returns boolean [B] flagged tiles + threshold.
    med = np.median(hf_scores)
    mad = np.median(np.abs(hf_scores - med)) + 1e-8
    thresh = med + k * 1.4826 * mad
    return hf_scores > thresh, thresh

def tile_metrics(true_ids, pred_ids):
    # Precision / recall / IoU at the tile level.
    tp = int((true_ids & pred_ids).sum())
    fp = int((~true_ids & pred_ids).sum())
    fn = int((true_ids & ~pred_ids).sum())
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec  = tp / (tp + fn) if tp + fn else float("nan")
    union = int((true_ids | pred_ids).sum())
    iou = tp / union if union else float("nan")
    return dict(precision=prec, recall=rec, tile_IoU=iou, tp=tp, fp=fp, fn=fn)

## 8. Full per-image pipeline + the 4-panel comparison
For one image we: pick the class -> attack random tiles -> tile the adversarial image -> score &
flag tiles -> grade against ground truth. The figure shows, left to right:

1. **Original** image.
2. **Attacked** image with the **TRUE** attacked tiles outlined (lime) -- what you see by eye.
3. **Blind HF-energy** heat per tile (what the detector measured).
4. **Detection result**: TRUE attacked tiles (lime, solid) vs **DETECTED** tiles (cyan, dashed) --
   when they overlap, the system caught the attack.

In [ ]:
# ============================================================
# CELL 8 - One-image pipeline + comparison figure
# ============================================================
def _outline_tiles(ax, ids, color, ls="-", lw=2.5):
    for k in np.where(ids)[0]:
        r, c = divmod(int(k), GRID)
        ax.add_patch(patches.Rectangle((c*TILE, r*TILE), TILE, TILE,
                     fill=False, edgecolor=color, linewidth=lw, linestyle=ls))

def run_one(path_or_url, n_attacked=3, epsilon=0.08, seed=0, show=True, name=""):
    x = load_image(path_or_url)                              # [1,3,SIZE,SIZE] CPU

    # clean prediction -> choose FGSM target class
    cls0, _ = batched_predict(x); cls0 = int(cls0[0])

    # attack a few random tiles -> ground truth
    attacked = random_tile_mask(n_attacked, seed=seed)
    x_adv = fgsm_attack(x, cls0, epsilon, attacked)

    # tile the adversarial image, score & flag every tile (blind)
    t_adv = tile_image(x_adv)
    hf = hf_energy_per_tile(t_adv)
    flagged, thr = detect_tiles(hf)
    m = tile_metrics(attacked, flagged)

    # did the global label flip?
    clsA, _ = batched_predict(x_adv); clsA = int(clsA[0])
    res = dict(name=name or os.path.basename(str(path_or_url)),
               before=idx2label[cls0], after=idx2label[clsA],
               fooled=clsA != cls0, n_attacked=int(attacked.sum()),
               n_flagged=int(flagged.sum()), **m)

    if show:
        orig_np, adv_np = to_np(x), to_np(x_adv)
        fig, ax = plt.subplots(1, 4, figsize=(18, 4.8))
        ax[0].imshow(orig_np); ax[0].set_title("1) Original")
        ax[1].imshow(adv_np); _outline_tiles(ax[1], attacked, "lime")
        ax[1].set_title("2) Attacked image\nlime = TRUE noisy tiles")
        ax[2].imshow(tile_grid_to_full(norm01(hf)), cmap="hot")
        ax[2].set_title("3) Blind HF-energy\n(per tile)")
        ax[3].imshow(adv_np)
        _outline_tiles(ax[3], attacked, "lime");  _outline_tiles(ax[3], flagged, "cyan", ls="--")
        ax[3].set_title("4) Detection\nlime = TRUE   cyan-- = DETECTED")
        for a in ax: a.axis("off")
        flag = "FOOLED" if res["fooled"] else "label held"
        fig.suptitle(f"{res['name']}  --  {res['before']} -> {res['after']}  [{flag}]   "
                     f"tile-IoU={m['tile_IoU']:.2f}  P={m['precision']:.2f}  R={m['recall']:.2f}",
                     y=1.04, fontsize=12)
        plt.tight_layout(); plt.show()
    return res

## 9. Run the whole workflow on the 10 images
Each image gets a different random set of attacked tiles (`seed=i`). Look at panel 2 to *see* where
the noise was injected, then panel 4 to see whether the blind detector flagged the same tiles.

In [ ]:
# ============================================================
# CELL 9 - Run on all 10 images (visual comparison per image)
# ============================================================
results = [run_one(p, n_attacked=3, epsilon=0.08, seed=i, show=True)
           for i, p in enumerate(images)]

## 10. Summary table -- how good is the detector?
Tile-level precision / recall / IoU per image, plus the averages and the global fool-rate.
High recall = it rarely misses a noisy tile; high precision = it rarely false-alarms a clean tile.

In [ ]:
# ============================================================
# CELL 10 - Summary table
# ============================================================
import pandas as pd
df = pd.DataFrame(results)
print(f"Global fool-rate: {df['fooled'].mean():.0%}")
print(f"Mean tile-IoU: {df['tile_IoU'].mean():.2f}   "
      f"Precision: {df['precision'].mean():.2f}   Recall: {df['recall'].mean():.2f}")
df[["name", "before", "after", "fooled", "n_attacked", "n_flagged",
    "precision", "recall", "tile_IoU"]]

## Recap & knobs
- **Dataset:** set `FOLDER` (Cell 4) to any Kaggle image dataset; 10 random images are sampled.
- **Attack visibility:** raise `epsilon` (e.g. `0.12`) to make the noise more obvious to the eye,
  lower it (`0.03`) to stress-test the detector.
- **How many tiles attacked:** `n_attacked` (1-16).
- **Detector sensitivity:** `detect_tiles(k=...)` -- lower `k` flags more tiles (higher recall,
  lower precision).
- **Granularity:** `GRID` (Cell 3) -- a finer grid localizes the attack better but costs more tiles.

**Next steps:** swap FGSM for a stronger iterative attack (PGD) to test robustness; drop the FGSM
step and run only the blind detector on real (possibly already-noisy) dataset images to clean them
before training.